In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ===================== USER SETTINGS =====================
MODE = "single"      
# Options:
# "single" -> one CSV file
# "batch"  -> all CSV files in a folder

SINGLE_FILE_PATH = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/drill_mech_depth_15aug20.csv"
BATCH_FOLDER_PATH = "/mnt/data/"   # change if needed

OUTPUT_FOLDER = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
# ==========================================================


DUMMY_VALUE = -999.25

# --------- MANUAL SCALE CONTROL ----------
MANUAL_X_SCALE = {
    # Example:
    # "GR": (0, 150),
    # "RHOB": (1.9, 2.9),
}
# -----------------------------------------


def get_auto_scale(series):
    clean = series.dropna()
    if len(clean) == 0:
        return (0, 1)
    vmin = np.percentile(clean, 2)
    vmax = np.percentile(clean, 98)
    return float(vmin), float(vmax)


def plot_well_log(csv_path, save_folder):
    df = pd.read_csv(csv_path)
    df = df.apply(pd.to_numeric, errors='coerce')

    df.replace(DUMMY_VALUE, np.nan, inplace=True)
    df.dropna(how="all", inplace=True)

    y = df.iloc[:, 0]
    features = df.columns[1:]
    n_tracks = len(features)

    fig, axes = plt.subplots(
        nrows=1, ncols=n_tracks,
        figsize=(3*n_tracks, 12),
        sharey=True
    )

    if n_tracks == 1:
        axes = [axes]

    # --- Store scaling report ---
    scale_report = {}

    for i, feature in enumerate(features):
        ax = axes[i]
        curve = df[feature]

        ax.plot(curve, y, linewidth=0.8)

        # Manual or auto scale
        if feature in MANUAL_X_SCALE:
            xmin, xmax = MANUAL_X_SCALE[feature]
            scale_type = "MANUAL"
        else:
            xmin, xmax = get_auto_scale(curve)
            scale_type = "AUTO"

        ax.set_xlim(xmin, xmax)

        ax.set_xlabel(feature)
        ax.grid(True)
        ax.invert_yaxis()

        scale_report[feature] = (scale_type, xmin, xmax)

    axes[0].set_ylabel(df.columns[0])
    plt.suptitle(os.path.basename(csv_path))

    os.makedirs(save_folder, exist_ok=True)
    out_name = os.path.splitext(os.path.basename(csv_path))[0] + "_welllog.png"
    out_path = os.path.join(save_folder, out_name)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

    # ===== PRINT SCALE REPORT =====
    print("\n--- X-AXIS SCALE REPORT ---")
    for feat, info in scale_report.items():
        print(f"{feat:15s} | {info[0]:6s} | xmin={info[1]:.3f} , xmax={info[2]:.3f}")
    print("----------------------------")
    print("Saved plot:", out_path)



# ===================== RUN =====================

if MODE == "single":
    plot_well_log(SINGLE_FILE_PATH, OUTPUT_FOLDER)

elif MODE == "batch":
    for file in os.listdir(BATCH_FOLDER_PATH):
        if file.lower().endswith(".csv"):
            plot_well_log(os.path.join(BATCH_FOLDER_PATH, file), OUTPUT_FOLDER)

print("All done.")


/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_62494/3275153989.py:77: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(xmin, xmax)



--- X-AXIS SCALE REPORT ---
HKLA            | AUTO   | xmin=2.248 , xmax=2.248
GR              | AUTO   | xmin=36.903 , xmax=87.737
SWOB            | AUTO   | xmin=-0.709 , xmax=4.685
RPM             | AUTO   | xmin=36.341 , xmax=91.198
TFLO            | AUTO   | xmin=177.745 , xmax=223.670
SPPA            | AUTO   | xmin=2993.222 , xmax=3957.960
ROP5            | AUTO   | xmin=32.659 , xmax=98.995
BLKP            | AUTO   | xmin=3.776 , xmax=46.608
STOR            | AUTO   | xmin=3.492 , xmax=7.161
CRPM            | AUTO   | xmin=17.140 , xmax=86.620
STICK           | AUTO   | xmin=12.000 , xmax=271.980
TRPM            | AUTO   | xmin=2640.000 , xmax=3420.000
SHKR            | AUTO   | xmin=0.000 , xmax=0.000
SHKRSK          | AUTO   | xmin=0.000 , xmax=0.000
TEMP_DNI        | AUTO   | xmin=145.400 , xmax=172.400
----------------------------
Saved plot: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/drill_mech_depth_15aug20_welllog.png
All done.
